In [1]:
# ---- edit if your folders differ ----
DRIVE_DIR      = "/content/drive/MyDrive/mustard_pp"          # where your zips are
UTTERANCE_ZIPS = "final_utterance_videos*.zip"                 # glob, all parts
CONTEXT_ZIPS   = "final_context_videos*.zip"                   # glob, all parts (optional)
USE_CONTEXT    = True                                          # False if your app has utterance audio only
MODEL_NAME     = "facebook/hubert-base-ls960"                  # or microsoft/wavlm-base-plus, facebook/wav2vec2-base
RUN_NAME       = "hubert_ctx_v2" if USE_CONTEXT else "hubert_base_v2"  # new folder; do not reuse the collapsed model
EPOCHS, BATCH_SIZE, GRAD_ACCUM = 20, 8, 2                      # on CUDA OOM: BATCH_SIZE=4, GRAD_ACCUM=4
# ------------------------------------
AUDIO_DIR = f"{DRIVE_DIR}/audio"            # extracted audio (persisted on Drive)
RUN_DIR   = f"{DRIVE_DIR}/runs/{RUN_NAME}"  # trained model (persisted on Drive)


## 2. GPU check, mount Drive, install

In [2]:
import subprocess, torch
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "NO GPU, enable one in Runtime > Change runtime type")
from google.colab import drive
drive.mount("/content/drive")
!pip -q install "transformers>=4.40" scikit-learn pandas librosa
!ffmpeg -version | head -1

GPU 0: Tesla T4 (UUID: GPU-eaf2703f-2571-fb7e-6838-00625c9a0edd)



KeyboardInterrupt: 

## 3. Write the training code
(The same files as in your Sim workspace folder `mustard-audio-sarcasm`.)

In [ ]:
!mkdir -p /content/sarcasm

In [ ]:
%%writefile /content/sarcasm/sarcasm_audio_model.py
"""Shared model + audio utilities for audio-only sarcasm detection (MUStARD++).

Used by both train_audio_sarcasm.py (training) and sarcasm_audio_infer.py (your app).
"""
import subprocess

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel

SAMPLE_RATE = 16000
MEDIA_EXTS = {".wav", ".mp3", ".flac", ".m4a", ".ogg", ".aac",
              ".mp4", ".mkv", ".avi", ".mov", ".webm"}


def load_audio(path, sr=SAMPLE_RATE):
    """Decode any audio/video file to mono float32 at `sr` Hz (ffmpeg, librosa fallback)."""
    try:
        raw = subprocess.run(
            ["ffmpeg", "-nostdin", "-v", "error", "-i", str(path),
             "-vn", "-f", "f32le", "-ac", "1", "-ar", str(sr), "-"],
            capture_output=True, check=True).stdout
        wav = np.frombuffer(raw, dtype=np.float32).copy()
    except (FileNotFoundError, subprocess.CalledProcessError):
        import librosa  # only needed if ffmpeg is missing/fails
        wav, _ = librosa.load(str(path), sr=sr, mono=True)
        wav = wav.astype(np.float32)
    if wav.size == 0:
        raise ValueError(f"No audio decoded from {path}")
    return wav


class SarcasmAudioModel(nn.Module):
    """SSL speech backbone (Wav2Vec2 / HuBERT / WavLM) + learned layer mix
    + attentive pooling + MLP classifier.

    use_context=True: the preceding dialogue (context clip) is encoded with the same
    backbone and fused with the utterance as [u, c, u*c] before the classifier.
    """

    def __init__(self, backbone=None, config=None, num_labels=2, dropout=0.2,
                 use_context=False):
        super().__init__()
        # LayerDrop is disabled: dropped layers vanish from hidden_states in recent
        # transformers versions, which would break the learned layer mix below.
        if config is not None:          # inference: build from saved config, weights loaded later
            config.layerdrop = 0.0
            self.backbone = AutoModel.from_config(config)
        else:                           # training: start from pretrained checkpoint
            self.backbone = AutoModel.from_pretrained(backbone, layerdrop=0.0)
        hidden = self.backbone.config.hidden_size
        n_layers = self.backbone.config.num_hidden_layers + 1  # + CNN/projection output
        self.use_context = use_context
        self.layer_weights = nn.Parameter(torch.zeros(n_layers))
        self.attn = nn.Sequential(nn.Linear(hidden, 128), nn.Tanh(), nn.Linear(128, 1))
        in_dim = hidden * 3 if use_context else hidden
        self.head = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(in_dim, 256), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(256, num_labels))

    def freeze_feature_encoder(self):
        fe = getattr(self.backbone, "feature_extractor", None)
        if fe is not None:
            for p in fe.parameters():
                p.requires_grad = False

    def encode(self, input_values, input_lengths, attention_mask=None):
        out = self.backbone(input_values, attention_mask=attention_mask,
                            output_hidden_states=True)
        hs = torch.stack(out.hidden_states, dim=0)                     # L,B,T,H
        w = F.softmax(self.layer_weights, dim=0).view(-1, 1, 1, 1)
        x = (w.to(hs.dtype) * hs).sum(0)                               # B,T,H
        T = x.size(1)
        frames = self.backbone._get_feat_extract_output_lengths(input_lengths)
        frames = frames.clamp(min=1, max=T)
        mask = torch.arange(T, device=x.device)[None, :] < frames[:, None]
        scores = self.attn(x).squeeze(-1).float().masked_fill(~mask, -1e4)
        alpha = scores.softmax(-1).unsqueeze(-1).to(x.dtype)
        return (alpha * x).sum(1)

    def forward(self, input_values, input_lengths, attention_mask=None,
                ctx_values=None, ctx_lengths=None, ctx_attention_mask=None, ctx_present=None):
        u = self.encode(input_values, input_lengths, attention_mask)
        if not self.use_context:
            return self.head(u)
        if ctx_values is None:
            c = torch.zeros_like(u)
        else:
            c = self.encode(ctx_values, ctx_lengths, ctx_attention_mask)
            if ctx_present is not None:      # clips with no context -> zero vector
                c = c * ctx_present.to(c.dtype)[:, None]
        return self.head(torch.cat([u, c, u * c], dim=-1))


Writing /content/sarcasm/sarcasm_audio_model.py


In [ ]:
%%writefile /content/sarcasm/train_audio_sarcasm.py
#!/usr/bin/env python3
"""Fine-tune Wav2Vec2 / HuBERT / WavLM for AUDIO-ONLY sarcasm detection on MUStARD++.

Example (Colab / any CUDA GPU):
  python train_audio_sarcasm.py --csv "mustard++_text.csv" \
      --media_dir /content/audio/utterances --context_dir /content/audio/context \
      --model facebook/hubert-base-ls960 --out runs/hubert_ctx
(--context_dir is optional; without it the model uses the utterance audio only.)
"""
import argparse
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_recall_fscore_support)
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from transformers import AutoFeatureExtractor, get_linear_schedule_with_warmup

from sarcasm_audio_model import MEDIA_EXTS, SAMPLE_RATE, SarcasmAudioModel, load_audio

LABELS = ["not_sarcastic", "sarcastic"]


# ----------------------------------------------------------------------------- data
def load_labels(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    df = df[df["KEY"].astype(str).str.endswith("_u")]          # target utterances only
    df = df[df["Sarcasm"].notna()].copy()
    df["label"] = df["Sarcasm"].astype(float).astype(int)
    for c in ("SCENE", "KEY", "SPEAKER", "SHOW"):
        df[c] = df[c].astype(str).str.strip()
    return df.reset_index(drop=True)


def index_media(media_dir, exclude):
    idx = {}
    for p in sorted(Path(media_dir).rglob("*")):
        if not p.is_file() or p.suffix.lower() not in MEDIA_EXTS:
            continue
        if exclude and any(e.lower() in str(p).lower() for e in exclude):
            continue
        idx.setdefault(p.stem, p)
    return idx


def match_media(row, idx):
    for cand in (row.KEY, f"{row.SCENE}_u", row.SCENE):
        if cand in idx:
            return idx[cand]
    return None


def find_context(scene, ctx_idx):
    """Context clip(s) for a scene: <SCENE>_c / <SCENE> / <SCENE>_context, or <SCENE>_c_00.. in order."""
    for cand in (f"{scene}_c", scene, f"{scene}_context", f"{scene}_ctx"):
        if cand in ctx_idx:
            return [ctx_idx[cand]]
    parts = sorted(k for k in ctx_idx if k.startswith(f"{scene}_c_"))
    return [ctx_idx[k] for k in parts]


def build_context_cache(df, context_dir, cache_dir, context_seconds):
    """Decode the context clip(s) per utterance and keep the LAST `context_seconds`
    (the dialogue right before the utterance). Returns a 0/1 'has_context' list."""
    cache_dir = Path(cache_dir)
    ctx_idx = index_media(context_dir, exclude=None)
    keep_n = int(context_seconds * SAMPLE_RATE)
    present, failed = [], []
    for row in df.itertuples():
        out = cache_dir / f"{row.KEY}__ctx.npy"
        if not out.exists():
            srcs = find_context(row.SCENE, ctx_idx)
            if not srcs:
                present.append(0)
                continue
            try:
                wav = np.concatenate([load_audio(p) for p in srcs])[-keep_n:]
                np.save(out, wav.astype(np.float16))
            except Exception as e:
                failed.append((row.SCENE, str(e)[:120]))
                present.append(0)
                continue
        present.append(1)
    print(f"Context audio found for {sum(present)}/{len(df)} utterances"
          + (f" ({len(failed)} failed to decode, e.g. {failed[:2]})" if failed else ""))
    if sum(present) == 0:
        raise SystemExit("No context clips matched. Expected names like 1_10004_c.mp4, "
                         "1_10004.mp4 or 1_10004_c_00.mp4 in --context_dir.")
    return present


def build_cache(df, media_dir, cache_dir, exclude):
    """Decode every utterance once to 16 kHz mono float16 .npy."""
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    idx = index_media(media_dir, exclude)
    keep, missing, failed = [], [], []
    for i, row in enumerate(df.itertuples()):
        out = cache_dir / f"{row.KEY}.npy"
        if not out.exists():
            src = match_media(row, idx)
            if src is None:
                missing.append(row.KEY)
                continue
            try:
                np.save(out, load_audio(src).astype(np.float16))
            except Exception as e:  # corrupt / silent file
                failed.append((row.KEY, str(e)[:120]))
                continue
        keep.append(i)
        if (i + 1) % 100 == 0:
            print(f"  cached {i + 1}/{len(df)}", flush=True)
    print(f"Audio ready for {len(keep)}/{len(df)} utterances "
          f"({len(missing)} missing, {len(failed)} failed to decode)")
    if missing:
        print("  e.g. missing:", missing[:5])
    if failed:
        print("  e.g. failed:", failed[:3])
    if not keep:
        raise SystemExit("No audio matched. Files must be named by KEY (e.g. 1_10004_u.mp4) "
                         "or SCENE (e.g. 1_10004.mp4).")
    return df.iloc[keep].reset_index(drop=True)


class UtteranceDataset(Dataset):
    def __init__(self, df, cache_dir, max_samples, train, use_context=False, context_drop_p=0.0):
        self.keys = df["KEY"].tolist()
        self.labels = df["label"].tolist()
        self.has_ctx = df["has_context"].tolist() if "has_context" in df else [0] * len(df)
        self.cache_dir = Path(cache_dir)
        self.max_samples = max_samples
        self.train = train
        self.use_context = use_context
        self.context_drop_p = context_drop_p

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, i):
        wav = np.load(self.cache_dir / f"{self.keys[i]}.npy").astype(np.float32)
        n = len(wav)
        if n > self.max_samples:
            start = (random.randint(0, n - self.max_samples) if self.train
                     else (n - self.max_samples) // 2)
            wav = wav[start:start + self.max_samples]
        if self.train:  # light augmentation: random gain + faint noise
            wav = wav * random.uniform(0.7, 1.3)
            if random.random() < 0.3:
                wav = wav + np.random.randn(len(wav)).astype(np.float32) * 0.003
        if not self.use_context:
            return wav, None, 0, self.labels[i]
        drop_ctx = self.train and random.random() < self.context_drop_p
        if self.has_ctx[i] and not drop_ctx:
            ctx = np.load(self.cache_dir / f"{self.keys[i]}__ctx.npy").astype(np.float32)
            if self.train:
                ctx = ctx * random.uniform(0.7, 1.3)
            return wav, ctx, 1, self.labels[i]
        return wav, np.zeros(SAMPLE_RATE, dtype=np.float32), 0, self.labels[i]


def make_collate(fe):
    use_mask = True  # always mask padded frames so silence does not dominate logits

    def enc(arrs):
        e = fe(list(arrs), sampling_rate=SAMPLE_RATE, padding=True, return_tensors="pt",
               return_attention_mask=use_mask)
        return e["input_values"], e.get("attention_mask"), torch.tensor([len(a) for a in arrs])

    def collate(batch):
        wavs, ctxs, present, labels = zip(*batch)
        b = {"labels": torch.tensor(labels, dtype=torch.long)}
        b["input_values"], b["attention_mask"], b["input_lengths"] = enc(wavs)
        if ctxs[0] is not None:
            b["ctx_values"], b["ctx_attention_mask"], b["ctx_lengths"] = enc(ctxs)
            b["ctx_present"] = torch.tensor(present, dtype=torch.float32)
        return b
    return collate


def forward_batch(model, b, device):
    mv = lambda k: b[k].to(device) if b.get(k) is not None else None
    return model(mv("input_values"), mv("input_lengths"), mv("attention_mask"),
                 mv("ctx_values"), mv("ctx_lengths"), mv("ctx_attention_mask"), mv("ctx_present"))


# ----------------------------------------------------------------------------- train / eval
def metrics(y, prob, thr=0.5):
    pred = (np.asarray(prob) >= thr).astype(int)
    p, r, f, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    return {"accuracy": accuracy_score(y, pred), "precision_macro": p, "recall_macro": r,
            "f1_macro": f, "f1_weighted": f1_score(y, pred, average="weighted", zero_division=0),
            "confusion_matrix": confusion_matrix(y, pred, labels=[0, 1]).tolist(),
            "n_pred_sarcastic": int(pred.sum()), "n_true_sarcastic": int(np.asarray(y).sum()),
            "mean_prob_sarcastic": float(np.mean(prob)), "threshold": float(thr)}


def best_threshold(y, prob):
    """Pick the sarcasm cutoff on the validation set (0.5 often labels everything as not sarcastic)."""
    y = np.asarray(y)
    prob = np.asarray(prob)
    grid = np.unique(np.concatenate([
        np.linspace(0.20, 0.80, 61),
        np.quantile(prob, np.linspace(0.10, 0.90, 17)),
    ]))
    best_t, best_m, best_f = 0.5, metrics(y, prob, 0.5), -1.0
    for t in grid:
        m = metrics(y, prob, float(t))
        # prefer higher macro-F1, then more balanced predicted class counts
        pred_rate = m["n_pred_sarcastic"] / max(len(y), 1)
        score = m["f1_macro"] - 0.15 * abs(pred_rate - float(y.mean()))
        if score > best_f:
            best_f, best_t, best_m = score, float(t), m
    return best_t, best_m


@torch.no_grad()
def predict(model, loader, device, amp_dtype):
    model.eval()
    probs, ys = [], []
    for b in loader:
        with torch.autocast(device.type, dtype=amp_dtype, enabled=amp_dtype is not None):
            logits = forward_batch(model, b, device)
        probs += logits.float().softmax(-1)[:, 1].cpu().tolist()
        ys += b["labels"].tolist()
    return np.array(ys), np.array(probs)


def class_weights(labels, device):
    counts = np.bincount(np.asarray(labels, dtype=int), minlength=2).astype(np.float64)
    counts = np.maximum(counts, 1.0)
    w = counts.sum() / (2.0 * counts)
    print(f"class weights (not_sarcastic, sarcastic) = {w.tolist()}  counts={counts.astype(int).tolist()}")
    return torch.tensor(w, dtype=torch.float32, device=device)


def init_classifier_bias(model, prior):
    prior = float(np.clip(prior, 0.05, 0.95))
    last = [m for m in model.head.modules() if isinstance(m, nn.Linear)][-1]
    with torch.no_grad():
        last.bias.zero_()
        last.bias[1] = math.log(prior / (1.0 - prior))
        last.bias[0] = math.log((1.0 - prior) / prior)


def train_one(args, train_df, val_df, test_df, fe, device, out_dir=None, tag=""):
    max_samples = int(args.max_seconds * SAMPLE_RATE)
    collate = make_collate(fe)
    g = torch.Generator().manual_seed(args.seed)
    uc = bool(args.context_dir)
    labels_np = train_df["label"].to_numpy()
    class_w = 1.0 / np.maximum(np.bincount(labels_np, minlength=2), 1)
    sampler = WeightedRandomSampler(
        torch.as_tensor(class_w[labels_np], dtype=torch.double),
        num_samples=len(labels_np), replacement=True, generator=g)
    tl = DataLoader(UtteranceDataset(train_df, args.cache_dir, max_samples, True, uc,
                                     context_drop_p=args.context_drop),
                    batch_size=args.batch_size, sampler=sampler, collate_fn=collate,
                    num_workers=args.num_workers, drop_last=True)
    mk = lambda d: DataLoader(UtteranceDataset(d, args.cache_dir, max_samples, False, uc),
                              batch_size=args.batch_size, collate_fn=collate,
                              num_workers=args.num_workers)
    vl, sl = mk(val_df), (mk(test_df) if test_df is not None and len(test_df) else None)

    model = SarcasmAudioModel(args.model, dropout=args.dropout, use_context=uc)
    model.freeze_feature_encoder()
    init_classifier_bias(model, float(labels_np.mean()))
    if args.gradient_checkpointing:
        model.backbone.gradient_checkpointing_enable()
    model.to(device)

    backbone_params = [p for p in model.backbone.parameters() if p.requires_grad]
    head_params = [p for n, p in model.named_parameters() if not n.startswith("backbone.")]
    opt = torch.optim.AdamW([
        {"params": backbone_params, "lr": args.lr_backbone},
        {"params": head_params, "lr": args.lr_head}], weight_decay=args.weight_decay)
    steps = math.ceil(len(tl) / args.grad_accum) * args.epochs
    sched = get_linear_schedule_with_warmup(opt, int(args.warmup_ratio * steps), steps)

    amp_dtype = None
    if device.type == "cuda" and not args.no_amp:
        amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    scaler = torch.amp.GradScaler(enabled=amp_dtype == torch.float16)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights(labels_np, device),
                                  label_smoothing=args.label_smoothing)

    best, best_state, best_thr, bad, history = -1.0, None, 0.5, 0, []
    for epoch in range(1, args.epochs + 1):
        frozen = epoch <= args.freeze_backbone_epochs  # warm up the head first
        for p in backbone_params:
            p.requires_grad = not frozen
        model.train()
        t0, running = time.time(), 0.0
        opt.zero_grad(set_to_none=True)
        for step, b in enumerate(tl, 1):
            with torch.autocast(device.type, dtype=amp_dtype, enabled=amp_dtype is not None):
                logits = forward_batch(model, b, device)
                loss = loss_fn(logits.float(), b["labels"].to(device)) / args.grad_accum
            scaler.scale(loss).backward()
            running += loss.item() * args.grad_accum
            if step % args.grad_accum == 0 or step == len(tl):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), args.max_grad_norm)
                scaler.step(opt)
                scaler.update()
                sched.step()
                opt.zero_grad(set_to_none=True)
        y, prob = predict(model, vl, device, amp_dtype)
        thr, m = best_threshold(y, prob)
        history.append({"epoch": epoch, "train_loss": running / len(tl), **m})
        print(f"{tag}epoch {epoch}/{args.epochs} loss={running / len(tl):.4f} "
              f"val_acc={m['accuracy']:.4f} val_f1={m['f1_macro']:.4f} "
              f"thr={thr:.3f} mean_p={m['mean_prob_sarcastic']:.3f} "
              f"pred_sarc={m['n_pred_sarcastic']}/{len(y)} cm={m['confusion_matrix']} "
              f"{'(backbone frozen) ' if frozen else ''}[{time.time() - t0:.0f}s]", flush=True)
        if m["f1_macro"] > best:
            best, bad, best_thr = m["f1_macro"], 0, thr
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        elif not frozen:
            bad += 1
            if bad >= args.patience:
                print(f"{tag}early stopping (no val F1 gain for {args.patience} epochs)")
                break

    model.load_state_dict(best_state)
    result = {"best_val_f1_macro": best, "threshold": best_thr, "history": history}
    if sl is not None:
        y, prob = predict(model, sl, device, amp_dtype)
        result["test"] = metrics(y, prob, best_thr)
        result["test_at_0.5"] = metrics(y, prob, 0.5)
        print(f"{tag}TEST acc={result['test']['accuracy']:.4f} "
              f"f1_macro={result['test']['f1_macro']:.4f} thr={best_thr:.3f} "
              f"mean_p={result['test']['mean_prob_sarcastic']:.3f} "
              f"cm={result['test']['confusion_matrix']}")
        result["_test_pred"] = pd.DataFrame({
            "KEY": test_df["KEY"].values, "SPEAKER": test_df["SPEAKER"].values,
            "SHOW": test_df["SHOW"].values, "label": y, "prob_sarcastic": prob,
            "pred": (prob >= best_thr).astype(int)})
    if out_dir is not None:
        save_model(model, fe, args, out_dir, result, best_thr)
    return result


def save_model(model, fe, args, out_dir, result, threshold):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), out_dir / "model.pt")
    model.backbone.config.save_pretrained(out_dir)
    fe.save_pretrained(out_dir)
    json.dump({"base_model": args.model, "labels": LABELS, "sample_rate": SAMPLE_RATE,
               "max_seconds": args.max_seconds, "dropout": args.dropout, "threshold": float(threshold),
               "use_context": bool(args.context_dir), "context_seconds": args.context_seconds,
               "best_val_f1_macro": result.get("best_val_f1_macro")},
              open(out_dir / "sarcasm_audio_config.json", "w"), indent=2)
    print(f"Saved model to {out_dir} (decision threshold={threshold:.3f})")


# ----------------------------------------------------------------------------- main
def parse_args():
    a = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    a.add_argument("--csv", required=True, help="mustard++_text.csv")
    a.add_argument("--media_dir", required=True, help="folder with utterance audio/video (searched recursively)")
    a.add_argument("--context_dir", default=None,
                   help="optional folder with context clips (final_context_videos or extracted audio); "
                        "enables utterance+context fusion")
    a.add_argument("--context_seconds", type=float, default=8.0,
                   help="keep the last N seconds of context (the lines right before the utterance)")
    a.add_argument("--context_drop", type=float, default=0.35,
                   help="during training, drop context this often so inference without context still works")
    a.add_argument("--out", default="runs/audio_sarcasm")
    a.add_argument("--cache_dir", default="audio_cache")
    a.add_argument("--exclude", nargs="*", default=["context"],
                   help="skip media whose path contains these substrings (context clips)")
    a.add_argument("--model", default="facebook/hubert-base-ls960",
                   help="facebook/wav2vec2-base | facebook/hubert-base-ls960 | microsoft/wavlm-base-plus | "
                        "facebook/hubert-large-ll60k | microsoft/wavlm-large ...")
    a.add_argument("--split", choices=["random", "cv", "show_holdout"], default="random")
    a.add_argument("--folds", type=int, default=5)
    a.add_argument("--holdout_show", default="FRIENDS", help="test show for --split show_holdout")
    a.add_argument("--val_size", type=float, default=0.15)
    a.add_argument("--test_size", type=float, default=0.15)
    a.add_argument("--epochs", type=int, default=15)
    a.add_argument("--batch_size", type=int, default=8)
    a.add_argument("--grad_accum", type=int, default=2)
    a.add_argument("--lr_backbone", type=float, default=2e-5)
    a.add_argument("--lr_head", type=float, default=1e-3)
    a.add_argument("--weight_decay", type=float, default=0.01)
    a.add_argument("--warmup_ratio", type=float, default=0.1)
    a.add_argument("--freeze_backbone_epochs", type=int, default=1)
    a.add_argument("--patience", type=int, default=5)
    a.add_argument("--max_seconds", type=float, default=10.0)
    a.add_argument("--dropout", type=float, default=0.2)
    a.add_argument("--label_smoothing", type=float, default=0.0)
    a.add_argument("--max_grad_norm", type=float, default=1.0)
    a.add_argument("--gradient_checkpointing", action="store_true", help="saves VRAM for large models")
    a.add_argument("--no_amp", action="store_true")
    a.add_argument("--num_workers", type=int, default=2)
    a.add_argument("--seed", type=int, default=42)
    a.add_argument("--limit", type=int, default=0, help="debug: use only N utterances")
    return a.parse_args()


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def main():
    args = parse_args()
    seed_all(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else
                                 "  WARNING: no GPU, this will be very slow"))
    df = load_labels(args.csv)
    if args.limit:
        df = df.groupby("label", group_keys=False).apply(
            lambda g: g.sample(min(len(g), args.limit // 2), random_state=args.seed)).reset_index(drop=True)
    print(f"{len(df)} labelled utterances: {df.label.value_counts().to_dict()}")
    df = build_cache(df, args.media_dir, args.cache_dir, args.exclude)
    if args.context_dir:
        df["has_context"] = build_context_cache(df, args.context_dir, args.cache_dir,
                                                args.context_seconds)
        if args.batch_size > 4 and "large" in args.model:
            print("Tip: context doubles memory; for large models use --batch_size 4 --gradient_checkpointing")
    fe = AutoFeatureExtractor.from_pretrained(args.model)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    summary = {"args": vars(args), "n_utterances": len(df)}

    if args.split in ("random", "show_holdout"):
        if args.split == "random":
            trval, test = train_test_split(df, test_size=args.test_size, stratify=df.label,
                                           random_state=args.seed)
        else:
            test = df[df.SHOW.str.upper() == args.holdout_show.upper()]
            trval = df[df.SHOW.str.upper() != args.holdout_show.upper()]
            if test.empty:
                raise SystemExit(f"No utterances for show {args.holdout_show}. Shows: {df.SHOW.unique()}")
        train, val = train_test_split(trval, test_size=args.val_size / (1 - args.test_size),
                                      stratify=trval.label, random_state=args.seed)
        print(f"split sizes: train={len(train)} val={len(val)} test={len(test)}")
        res = train_one(args, train, val, test, fe, device, out_dir=out)
        res.pop("_test_pred").to_csv(out / "test_predictions.csv", index=False)
        summary.update(res)
    else:
        skf = StratifiedKFold(n_splits=args.folds, shuffle=True, random_state=args.seed)
        folds = []
        for k, (tr_i, te_i) in enumerate(skf.split(df, df.label), 1):
            trval, test = df.iloc[tr_i], df.iloc[te_i]
            train, val = train_test_split(trval, test_size=0.1, stratify=trval.label,
                                          random_state=args.seed)
            res = train_one(args, train, val, test, fe, device, out_dir=None, tag=f"[fold {k}] ")
            res.pop("_test_pred").to_csv(out / f"test_predictions_fold{k}.csv", index=False)
            folds.append(res)
        for key in ("accuracy", "f1_macro", "precision_macro", "recall_macro"):
            vals = [f["test"][key] for f in folds]
            summary[f"cv_{key}_mean"], summary[f"cv_{key}_std"] = float(np.mean(vals)), float(np.std(vals))
        summary["folds"] = folds
        print(f"CV ({args.folds} folds) accuracy={summary['cv_accuracy_mean']:.4f}±{summary['cv_accuracy_std']:.4f} "
              f"f1_macro={summary['cv_f1_macro_mean']:.4f}±{summary['cv_f1_macro_std']:.4f}")
        print("CV only estimates performance. Re-run with --split random to save a model for your app.")

    json.dump(summary, open(out / "metrics.json", "w"), indent=2, default=float)
    print(f"Metrics written to {out / 'metrics.json'}")


if __name__ == "__main__":
    main()


Writing /content/sarcasm/train_audio_sarcasm.py


In [ ]:
%%writefile /content/sarcasm/sarcasm_audio_infer.py
"""Load the trained audio sarcasm model in your app.

    from sarcasm_audio_infer import SarcasmAudioDetector
    det = SarcasmAudioDetector("runs/hubert_ctx")
    det.predict("utterance.wav")                          # utterance only
    det.predict("utterance.wav", context="previous_lines.wav")   # if trained with --context_dir
    det.predict(np_array, sr=44100, context=ctx_array, context_sr=44100)
    # -> {'label': 'sarcastic', 'is_sarcastic': True, 'sarcasm_probability': 0.83, 'used_context': True}

CLI:  python sarcasm_audio_infer.py MODEL_DIR utterance.wav [context.wav]
"""
import json
import sys
from pathlib import Path

import numpy as np
import torch
from transformers import AutoConfig, AutoFeatureExtractor

from sarcasm_audio_model import SAMPLE_RATE, SarcasmAudioModel, load_audio


def _to_wav(audio, sr):
    if isinstance(audio, (str, Path)):
        return load_audio(audio)
    wav = np.asarray(audio, dtype=np.float32)
    if wav.ndim > 1:
        wav = wav.mean(axis=-1 if wav.shape[-1] <= 8 else 0)
    if sr and sr != SAMPLE_RATE:
        import librosa
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SAMPLE_RATE)
    return wav.astype(np.float32)


class SarcasmAudioDetector:
    def __init__(self, model_dir, device=None):
        model_dir = Path(model_dir)
        self.cfg = json.load(open(model_dir / "sarcasm_audio_config.json"))
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        self.fe = AutoFeatureExtractor.from_pretrained(model_dir)
        self.use_context = bool(self.cfg.get("use_context", False))
        self.model = SarcasmAudioModel(config=AutoConfig.from_pretrained(model_dir),
                                       dropout=self.cfg.get("dropout", 0.2),
                                       use_context=self.use_context)
        state = torch.load(model_dir / "model.pt", map_location="cpu", weights_only=True)
        self.model.load_state_dict(state)
        self.model.to(self.device).eval()
        self.window = int(self.cfg.get("max_seconds", 10) * SAMPLE_RATE)
        self.ctx_keep = int(self.cfg.get("context_seconds", 8) * SAMPLE_RATE)
        self.threshold = float(self.cfg.get("threshold", 0.5))
        self._mask = True  # always mask padding; a silent 0.5 cutoff was collapsing to not_sarcastic

    def _enc(self, arrs):
        e = self.fe(arrs, sampling_rate=SAMPLE_RATE, padding=True, return_tensors="pt",
                    return_attention_mask=self._mask)
        am = e.get("attention_mask")
        return (e["input_values"].to(self.device),
                torch.tensor([len(a) for a in arrs], device=self.device),
                am.to(self.device) if am is not None else None)

    @torch.no_grad()
    def predict(self, audio, sr=None, context=None, context_sr=None):
        wav = _to_wav(audio, sr)
        # long clips: score overlapping windows and average
        hop = self.window // 2
        chunks = [wav[i:i + self.window] for i in range(0, max(len(wav) - hop, 1), hop)] \
            if len(wav) > self.window else [wav]
        u = self._enc(chunks)
        used_ctx = False
        if self.use_context and context is not None:
            ctx = _to_wav(context, context_sr)[-self.ctx_keep:]
            c = self._enc([ctx] * len(chunks))
            present = torch.ones(len(chunks), device=self.device)
            logits = self.model(*u, *c, present)
            used_ctx = True
        else:
            logits = self.model(*u)   # context-trained model falls back to zero context
        p = float(logits.softmax(-1)[:, 1].mean())
        return {"label": "sarcastic" if p >= self.threshold else "not_sarcastic",
                "is_sarcastic": p >= self.threshold, "sarcasm_probability": round(p, 4),
                "used_context": used_ctx}


if __name__ == "__main__":
    if len(sys.argv) < 3:
        sys.exit(__doc__)
    det = SarcasmAudioDetector(sys.argv[1])
    print(sys.argv[2], det.predict(sys.argv[2], context=sys.argv[3] if len(sys.argv) > 3 else None))


Writing /content/sarcasm/sarcasm_audio_infer.py


In [ ]:
%%writefile /content/sarcasm/extract_audio.py
#!/usr/bin/env python3
"""Extract 16 kHz mono audio from MUStARD++ video zips or folders (run locally or on Colab).

Shrinks ~2 GB of video to a few hundred MB of audio, keeping each file's name
(e.g. final_context_videos/1_10004_c.mp4 -> OUT/1_10004_c.flac) so the training
script can match it to the labels.

  python extract_audio.py final_context_videos-*.zip --out audio/context
  python extract_audio.py final_utterance_videos-*.zip --out audio/utterances
  python extract_audio.py /path/to/video_folder --out audio/utterances --format wav

Needs ffmpeg on PATH (Colab has it; Windows: `winget install ffmpeg`).
"""
import argparse
import os
import shutil
import subprocess
import tempfile
import zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

VIDEO_AUDIO_EXTS = {".mp4", ".mkv", ".avi", ".mov", ".webm", ".m4a", ".mp3", ".wav", ".flac", ".aac", ".ogg"}


def ffmpeg_extract(src, dst, sr):
    tmp = dst.with_suffix(dst.suffix + ".part")
    subprocess.run(["ffmpeg", "-nostdin", "-v", "error", "-y", "-i", str(src), "-vn",
                    "-ac", "1", "-ar", str(sr), "-f", dst.suffix[1:], str(tmp)],
                   check=True, capture_output=True)
    tmp.replace(dst)


def jobs_for(source):
    p = Path(source)
    if p.suffix.lower() == ".zip":
        with zipfile.ZipFile(p) as z:
            for name in z.namelist():
                if Path(name).suffix.lower() in VIDEO_AUDIO_EXTS and not Path(name).name.startswith("._"):
                    yield ("zip", str(p), name, Path(name).stem)
    else:
        for f in sorted(p.rglob("*")):
            if f.is_file() and f.suffix.lower() in VIDEO_AUDIO_EXTS:
                yield ("file", str(f), None, f.stem)


def run_job(job, out_dir, fmt, sr):
    kind, src, member, stem = job
    dst = out_dir / f"{stem}.{fmt}"
    if dst.exists():
        return "skipped", stem
    if kind == "file":
        ffmpeg_extract(src, dst, sr)
    else:  # copy the member to a temp file (mp4 needs a seekable input)
        with zipfile.ZipFile(src) as z, z.open(member) as fin, \
                tempfile.NamedTemporaryFile(suffix=Path(member).suffix, delete=False) as tmp:
            shutil.copyfileobj(fin, tmp, 1 << 20)
        try:
            ffmpeg_extract(tmp.name, dst, sr)
        finally:
            os.unlink(tmp.name)
    return "ok", stem


def main():
    a = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    a.add_argument("sources", nargs="+", help="zip files and/or folders of videos")
    a.add_argument("--out", required=True)
    a.add_argument("--format", choices=["flac", "wav"], default="flac", help="flac = lossless and ~2x smaller")
    a.add_argument("--sr", type=int, default=16000)
    a.add_argument("--workers", type=int, default=os.cpu_count() or 4)
    args = a.parse_args()
    if shutil.which("ffmpeg") is None:
        raise SystemExit("ffmpeg not found on PATH")
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    jobs, seen, dups = [], set(), 0
    for s in args.sources:
        for j in jobs_for(s):
            if j[3] in seen:
                dups += 1
                continue
            seen.add(j[3])
            jobs.append(j)
    print(f"{len(jobs)} media files found" + (f" ({dups} duplicate names skipped)" if dups else ""))
    counts, failed = {"ok": 0, "skipped": 0}, []
    with ThreadPoolExecutor(args.workers) as ex:
        futs = {ex.submit(run_job, j, out, args.format, args.sr): j for j in jobs}
        for i, f in enumerate(as_completed(futs), 1):
            try:
                status, _ = f.result()
                counts[status] += 1
            except Exception as e:
                err = getattr(e, "stderr", b"") or str(e).encode()
                failed.append((futs[f][3], err.decode(errors="ignore").strip()[:150]))
            if i % 100 == 0 or i == len(jobs):
                print(f"  {i}/{len(jobs)} done", flush=True)
    print(f"Extracted {counts['ok']}, already present {counts['skipped']}, failed {len(failed)} -> {out}")
    for stem, err in failed[:10]:
        print(f"  FAILED {stem}: {err}")


if __name__ == "__main__":
    main()


Writing /content/sarcasm/extract_audio.py


In [ ]:
# MUStARD++ labels (official CSV from the MUStARD++ GitHub repo)
!curl -sL "https://raw.githubusercontent.com/cfiltnlp/MUStARD_Plus_Plus/main/mustard%2B%2B_text.csv" -o "/content/sarcasm/mustard++_text.csv"
import pandas as pd
df = pd.read_csv("/content/sarcasm/mustard++_text.csv"); df.columns = [c.strip() for c in df.columns]
u = df[df.KEY.astype(str).str.endswith("_u")]
print(len(u), "labelled utterances", u.Sarcasm.value_counts().to_dict())

1201 labelled utterances {1.0: 601, 0.0: 600}


## 4. Find your zips and copy them to the fast local disk

In [ ]:
import glob, os, shutil
utt_zips = sorted(glob.glob(f"{DRIVE_DIR}/{UTTERANCE_ZIPS}"))
ctx_zips = sorted(glob.glob(f"{DRIVE_DIR}/{CONTEXT_ZIPS}"))
print("utterance zips:", [os.path.basename(z) for z in utt_zips])
print("context zips:  ", [os.path.basename(z) for z in ctx_zips])
assert utt_zips, f"No {UTTERANCE_ZIPS} in {DRIVE_DIR}. Upload the utterance video zips (required)."
if USE_CONTEXT and not ctx_zips:
    print("No context zips found, training utterance-only."); USE_CONTEXT = False
os.makedirs("/content/zips", exist_ok=True)
def local(zs):
    out = []
    for z in zs:
        dst = f"/content/zips/{os.path.basename(z)}"
        if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(z):
            print("copying", os.path.basename(z)); shutil.copy(z, dst)
        out.append(dst)
    return out
utt_local = local(utt_zips)
ctx_local = local(ctx_zips) if USE_CONTEXT else []

utterance zips: []
context zips:   []


AssertionError: No final_utterance_videos*.zip in /content/drive/MyDrive/mustard_pp. Upload the utterance video zips (required).

In [ ]:
import glob
import os
import shutil

VIDEO_DIR = os.path.join(DRIVE_DIR, "videos")

UTTERANCE_ZIPS = "final_utterance_videos-*.zip"
CONTEXT_ZIPS = "final_context_videos-*.zip"

utt_zips = sorted(glob.glob(os.path.join(VIDEO_DIR, UTTERANCE_ZIPS)))
ctx_zips = sorted(glob.glob(os.path.join(VIDEO_DIR, CONTEXT_ZIPS)))

print("Utterance zips:", [os.path.basename(z) for z in utt_zips])
print("Context zips:", [os.path.basename(z) for z in ctx_zips])

assert utt_zips, "Utterance ZIP files not found!"

if USE_CONTEXT and not ctx_zips:
    print("No context zips found, training utterance-only.")
    USE_CONTEXT = False

os.makedirs("/content/zips", exist_ok=True)

def local(zs):
    out = []
    for z in zs:
        dst = f"/content/zips/{os.path.basename(z)}"
        if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(z):
            print("Copying", os.path.basename(z))
            shutil.copy(z, dst)
        out.append(dst)
    return out

utt_local = local(utt_zips)
ctx_local = local(ctx_zips) if USE_CONTEXT else []

Utterance zips: []
Context zips: []


AssertionError: Utterance ZIP files not found!

In [ ]:
import os
import glob

print("DRIVE_DIR:", DRIVE_DIR)
print("Does DRIVE_DIR exist?", os.path.exists(DRIVE_DIR))

print("\nFolders inside DRIVE_DIR:")
if os.path.exists(DRIVE_DIR):
    print(os.listdir(DRIVE_DIR))

print("\nSearching for video ZIP files:")
all_zips = glob.glob("/content/drive/MyDrive/**/*.zip", recursive=True)

for z in all_zips:
    if "utterance" in os.path.basename(z).lower() or "context" in os.path.basename(z).lower():
        print(z)

DRIVE_DIR: /content/drive/MyDrive/mustard_pp
Does DRIVE_DIR exist? True

Folders inside DRIVE_DIR:
['audio', 'runs']

Searching for video ZIP files:
/content/drive/MyDrive/videos/final_context_videos-20261013T201657Z-1-001.zip
/content/drive/MyDrive/videos/final_utterance_videos-20261020T090313Z-1-001.zip


In [ ]:
import glob
import os
import shutil

VIDEO_DIR = "/content/drive/MyDrive/videos"

UTTERANCE_ZIPS = "final_utterance_videos-*.zip"
CONTEXT_ZIPS = "final_context_videos-*.zip"

utt_zips = sorted(glob.glob(os.path.join(VIDEO_DIR, UTTERANCE_ZIPS)))
ctx_zips = sorted(glob.glob(os.path.join(VIDEO_DIR, CONTEXT_ZIPS)))

print("Utterance zips:", [os.path.basename(z) for z in utt_zips])
print("Context zips:", [os.path.basename(z) for z in ctx_zips])

assert utt_zips, "Utterance ZIP files not found!"

if USE_CONTEXT and not ctx_zips:
    print("No context zips found, training utterance-only.")
    USE_CONTEXT = False

os.makedirs("/content/zips", exist_ok=True)

def local(zs):
    out = []
    for z in zs:
        dst = f"/content/zips/{os.path.basename(z)}"
        if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(z):
            print("Copying", os.path.basename(z))
            shutil.copy(z, dst)
        out.append(dst)
    return out

utt_local = local(utt_zips)
ctx_local = local(ctx_zips) if USE_CONTEXT else []

print("Utterance files copied:", len(utt_local))
print("Context files copied:", len(ctx_local))

Utterance zips: ['final_utterance_videos-20261020T090313Z-1-001.zip']
Context zips: ['final_context_videos-20261013T201657Z-1-001.zip']
Copying final_utterance_videos-20261020T090313Z-1-001.zip
Copying final_context_videos-20261013T201657Z-1-001.zip
Utterance files copied: 1
Context files copied: 1


In [ ]:
import os

print("Drive folder exists:", os.path.exists(DRIVE_DIR))
print("DRIVE_DIR:", DRIVE_DIR)

print("Files inside mustard_pp:")
print(os.listdir(DRIVE_DIR))

print("Files inside final_utterance_videos:")
folder = os.path.join(DRIVE_DIR, "final_utterance_videos")

if os.path.exists(folder):
    print(os.listdir(folder))
else:
    print("Folder does not exist!")

Drive folder exists: True
DRIVE_DIR: /content/drive/MyDrive/mustard_pp
Files inside mustard_pp:
['audio', 'runs']
Files inside final_utterance_videos:
Folder does not exist!


In [ ]:
import glob
import os
import shutil

VIDEO_DIR = os.path.join(DRIVE_DIR, "videos")

UTTERANCE_ZIPS = "final_utterance_videos-*.zip"
CONTEXT_ZIPS = "final_context_videos-*.zip"

utt_zips = sorted(glob.glob(os.path.join(VIDEO_DIR, UTTERANCE_ZIPS)))
ctx_zips = sorted(glob.glob(os.path.join(VIDEO_DIR, CONTEXT_ZIPS)))

print("Utterance zips:", [os.path.basename(z) for z in utt_zips])
print("Context zips:", [os.path.basename(z) for z in ctx_zips])

assert utt_zips, "Utterance ZIP files not found!"

if USE_CONTEXT and not ctx_zips:
    print("No context zips found, training utterance-only.")
    USE_CONTEXT = False

os.makedirs("/content/zips", exist_ok=True)

def local(zs):
    out = []
    for z in zs:
        dst = f"/content/zips/{os.path.basename(z)}"
        if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(z):
            print("Copying", os.path.basename(z))
            shutil.copy(z, dst)
        out.append(dst)
    return out

utt_local = local(utt_zips)
ctx_local = local(ctx_zips) if USE_CONTEXT else []

Utterance zips: []
Context zips: []


AssertionError: Utterance ZIP files not found!

In [ ]:
VIDEO_DIR = "/content/drive/MyDrive/videos"

In [ ]:
import glob
import os
import shutil

VIDEO_DIR = "/content/drive/MyDrive/videos"

UTTERANCE_ZIPS = "final_utterance_videos-*.zip"
CONTEXT_ZIPS = "final_context_videos-*.zip"

utt_zips = sorted(glob.glob(os.path.join(VIDEO_DIR, UTTERANCE_ZIPS)))
ctx_zips = sorted(glob.glob(os.path.join(VIDEO_DIR, CONTEXT_ZIPS)))

print("Utterance zips:", [os.path.basename(z) for z in utt_zips])
print("Context zips:", [os.path.basename(z) for z in ctx_zips])

assert utt_zips, "Utterance ZIP files not found!"

if USE_CONTEXT and not ctx_zips:
    print("No context zips found, training utterance-only.")
    USE_CONTEXT = False

os.makedirs("/content/zips", exist_ok=True)

def local(zs):
    out = []
    for z in zs:
        dst = f"/content/zips/{os.path.basename(z)}"
        if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(z):
            print("Copying", os.path.basename(z))
            shutil.copy(z, dst)
        out.append(dst)
    return out

utt_local = local(utt_zips)
ctx_local = local(ctx_zips) if USE_CONTEXT else []

print("Utterance copied:", len(utt_local))
print("Context copied:", len(ctx_local))

Utterance zips: ['final_utterance_videos-20261020T090313Z-1-001.zip']
Context zips: ['final_context_videos-20261013T201657Z-1-001.zip']
Utterance copied: 1
Context copied: 1


In [ ]:
import os

RUN_DIR = "/content/drive/MyDrive/mustard_pp/runs"

for root, dirs, files in os.walk(RUN_DIR):
    for file in files:
        if file.endswith((".pt", ".pth", ".bin", ".safetensors")):
            print(os.path.join(root, file))

## 5. Extract audio from the videos (16 kHz mono FLAC, saved to Drive)

In [ ]:
import shlex
q = lambda xs: " ".join(shlex.quote(x) for x in xs)
!cd /content/sarcasm && python extract_audio.py {q(utt_local)} --out {shlex.quote(AUDIO_DIR + "/utterances")}
assert _exit_code == 0, "Utterance audio extraction failed, see the log above"
if USE_CONTEXT:
    !cd /content/sarcasm && python extract_audio.py {q(ctx_local)} --out {shlex.quote(AUDIO_DIR + "/context")}
    assert _exit_code == 0, "Context audio extraction failed, see the log above"
for d in ["utterances", "context"]:
    p = f"{AUDIO_DIR}/{d}"
    if os.path.isdir(p): print(d, len(os.listdir(p)), "audio files")
n_utt = len(os.listdir(f"{AUDIO_DIR}/utterances"))
if n_utt < 1150:
    print(f"WARNING: only {n_utt} utterance clips (MUStARD++ has 1,202). Are all zip parts in {DRIVE_DIR}?")

1203 media files found
  100/1203 done
  200/1203 done
  300/1203 done
  400/1203 done
  500/1203 done
  600/1203 done
  700/1203 done
  800/1203 done
  900/1203 done
  1000/1203 done
  1100/1203 done
  1200/1203 done
  1203/1203 done
Extracted 0, already present 1203, failed 0 -> /content/drive/MyDrive/mustard_pp/audio/utterances
1212 media files found
  100/1212 done
  200/1212 done
  300/1212 done
  400/1212 done
  500/1212 done
  600/1212 done
  700/1212 done
  800/1212 done
  900/1212 done
  1000/1212 done
  1100/1212 done
  1200/1212 done
  1212/1212 done
Extracted 0, already present 1212, failed 0 -> /content/drive/MyDrive/mustard_pp/audio/context
utterances 1203 audio files
context 1212 audio files


## 6. Train
Prints validation accuracy/F1, the **tuned sarcasm threshold**, predicted sarcastic count, and confusion matrix each epoch. Best epoch is kept (early stopping starts after the backbone unfreezes).


In [ ]:
ctx_flag = f"--context_dir {shlex.quote(AUDIO_DIR + '/context')}" if USE_CONTEXT else ""
!cd /content/sarcasm && python train_audio_sarcasm.py --csv "mustard++_text.csv" \
    --media_dir {shlex.quote(AUDIO_DIR + "/utterances")} {ctx_flag} \
    --cache_dir /content/audio_cache --model {MODEL_NAME} --out {shlex.quote(RUN_DIR)} \
    --epochs {EPOCHS} --batch_size {BATCH_SIZE} --grad_accum {GRAD_ACCUM}
assert _exit_code == 0, "Training failed, see the log above (CUDA out of memory? set BATCH_SIZE=4, GRAD_ACCUM=4 in Settings)"

Device: cuda (Tesla T4)
1201 labelled utterances: {1: 601, 0: 600}
  cached 100/1201
  cached 200/1201
  cached 300/1201
  cached 400/1201
  cached 500/1201
  cached 600/1201
  cached 700/1201
  cached 800/1201
  cached 900/1201
  cached 1000/1201
  cached 1100/1201
  cached 1200/1201
Audio ready for 1201/1201 utterances (0 missing, 0 failed to decode)
Context audio found for 1201/1201 utterances
preprocessor_config.json: 100% 213/213 [00:00<00:00, 938kB/s]
split sizes: train=840 val=180 test=181
config.json: 100% 1.39k/1.39k [00:00<00:00, 4.25MB/s]

pytorch_model.bin: downloading bytes:  14% 51.1M/378M [00:01<00:05, 62.6MB/s, 2.17MB/s  ]
pytorch_model.bin: downloading bytes:  33% 124M/378M [00:01<00:02, 123MB/s, 9.78MB/s  ]  
pytorch_model.bin: downloading bytes:  41% 156M/378M [00:01<00:01, 133MB/s, 12.7MB/s  ]
pytorch_model.bin: downloading bytes:  47% 176M/378M [00:01<00:01, 144MB/s, 14.5MB/s  ]
pytorch_model.bin: reconstructing file:  28% 106M/378M [00:02<00:03, 70.7MB/s, 8.38MB/s

## 7. Results

In [ ]:
import json
m = json.load(open(f"{RUN_DIR}/metrics.json"))
t = m["test"]
cfg = json.load(open(f"{RUN_DIR}/sarcasm_audio_config.json"))
print(f"decision threshold {cfg['threshold']:.3f} (not a raw 0.5 cutoff)")
print(f"TEST accuracy {t['accuracy']:.3f} | macro-F1 {t['f1_macro']:.3f} | precision {t['precision_macro']:.3f} | recall {t['recall_macro']:.3f}")
print(f"mean P(sarcastic)={t.get('mean_prob_sarcastic')}  predicted sarcastic={t.get('n_pred_sarcastic')}  true sarcastic={t.get('n_true_sarcastic')}")
print("confusion matrix [[TN, FP], [FN, TP]]:", t["confusion_matrix"])
if t.get("n_pred_sarcastic", 1) == 0:
    print("WARNING: still predicting 0 sarcastic clips — re-run training, do not ship this checkpoint.")
pd.DataFrame(m["history"])[[c for c in ["epoch", "train_loss", "accuracy", "f1_macro", "threshold", "mean_prob_sarcastic", "n_pred_sarcastic"] if c in m["history"][0]]]


decision threshold 0.510 (not a raw 0.5 cutoff)
TEST accuracy 0.696 | macro-F1 0.695 | precision 0.698 | recall 0.696
mean P(sarcastic)=0.5338839555200374  predicted sarcastic=100  true sarcastic=91
confusion matrix [[TN, FP], [FN, TP]]: [[58, 32], [23, 68]]


,epoch,train_loss,accuracy,f1_macro,threshold,mean_prob_sarcastic,n_pred_sarcastic
0,1,0.678275,0.638889,0.633906,0.540000,0.574291,111
1,2,0.609807,0.688889,0.686995,0.640000,0.524930,76
2,3,0.545947,0.683333,0.683245,0.560000,0.540688,87
3,4,0.532438,0.727778,0.727365,0.510000,0.532087,97
4,5,0.476717,0.711111,0.710539,0.450000,0.488078,98
5,6,0.412440,0.716667,0.716448,0.570000,0.549277,95
6,7,0.349213,0.705556,0.705474,0.690000,0.559160,87
7,8,0.292625,0.688889,0.688889,0.779315,0.593621,90
8,9,0.260577,0.688889,0.688543,0.490000,0.489346,84


## 8. Try the trained model

In [ ]:
import sys; sys.path.insert(0, "/content/sarcasm")
from sarcasm_audio_infer import SarcasmAudioDetector
det = SarcasmAudioDetector(RUN_DIR)
preds = pd.read_csv(f"{RUN_DIR}/test_predictions.csv").head(5)
for k, y in zip(preds.KEY, preds.label):
    utt = glob.glob(f"{AUDIO_DIR}/utterances/{k}.*") or glob.glob(f"{AUDIO_DIR}/utterances/{k[:-2]}.*")
    ctx = (glob.glob(f"{AUDIO_DIR}/context/{k[:-2]}_c.*") or [None])[0] if USE_CONTEXT else None
    if utt: print(k, "true:", "sarcastic" if y else "not_sarcastic", "->", det.predict(utt[0], context=ctx))

1_S10E18_164_u true: sarcastic -> {'label': 'sarcastic', 'is_sarcastic': True, 'sarcasm_probability': 0.9644, 'used_context': True}
1_S09E03_125_u true: sarcastic -> {'label': 'sarcastic', 'is_sarcastic': True, 'sarcasm_probability': 0.9637, 'used_context': True}
2_109_u true: sarcastic -> {'label': 'sarcastic', 'is_sarcastic': True, 'sarcasm_probability': 0.9356, 'used_context': True}
1_S12E03_239_u true: not_sarcastic -> {'label': 'not_sarcastic', 'is_sarcastic': False, 'sarcasm_probability': 0.0286, 'used_context': True}
2_416_u true: sarcastic -> {'label': 'sarcastic', 'is_sarcastic': True, 'sarcasm_probability': 0.9457, 'used_context': True}


In [ ]:
!pip install gTTS librosa soundfile -q

from gtts import gTTS
import librosa
import soundfile as sf
from IPython.display import Audio, display

text = "Oh wow, that's just fantastic! I absolutely love waiting for three hours."

tts = gTTS(text=text, lang="en", slow=False)
tts.save("/content/sarcastic_test.mp3")

audio, sr = librosa.load("/content/sarcastic_test.mp3", sr=16000, mono=True)

sf.write("/content/sarcastic_test.wav", audio, 16000)

display(Audio(audio, rate=16000))

print("Audio saved at: /content/sarcastic_test.wav")
print("Sample rate:", 16000)
print("Duration:", len(audio) / 16000, "seconds")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.33.0 requires click<9.0.0,>=8.4.2, but you have click 8.1.8 which is incompatible.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
spacy 3.8.18 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.


Audio saved at: /content/sarcastic_test.wav
Sample rate: 16000
Duration: 6.072 seconds


In [ ]:
from google.colab import files

uploaded = files.upload()
audio_path = next(iter(uploaded.keys()))

print("Uploaded:", audio_path)

Saving sarcastic_test.wav to sarcastic_test (1).wav
Uploaded: sarcastic_test (1).wav


In [ ]:
import os

RUN_DIR = "/content/drive/MyDrive/mustard_pp/runs/hubert_ctx_v2"

for root, dirs, filenames in os.walk(RUN_DIR):
    for filename in filenames:
        print(os.path.join(root, filename))

In [ ]:
import librosa
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

audio, sr = librosa.load(audio_path, sr=16000, mono=True)

audio = np.asarray(audio, dtype=np.float32)

print("Sample rate:", sr)
print("Audio duration:", len(audio) / sr)
print("Audio shape:", audio.shape)

display(Audio(audio, rate=sr))

Sample rate: 16000
Audio duration: 6.1546875
Audio shape: (98475,)


In [ ]:
import os

RUN_DIR = "/content/drive/MyDrive/mustard_pp/runs/hubert_ctx_v2"

print("Model directory exists:", os.path.exists(RUN_DIR))

for root, dirs, files in os.walk(RUN_DIR):
    for file in files:
        print(os.path.join(root, file))

Model directory exists: True


In [ ]:
import os

RUN_DIR = "/content/drive/MyDrive/mustard_pp/runs/hubert_ctx_v2"

for root, dirs, files in os.walk(RUN_DIR):
    print("\nFolder:", root)

    if not files:
        print("No files found")

    for file in files:
        path = os.path.join(root, file)
        print(file, "| Size:", round(os.path.getsize(path) / (1024 * 1024), 2), "MB")


Folder: /content/drive/MyDrive/mustard_pp/runs/hubert_ctx_v2
No files found


In [ ]:
import os

RUN_DIR = "/content/drive/MyDrive/mustard_pp/runs/hubert_ctx_v2"

for root, dirs, filenames in os.walk(RUN_DIR):
    for filename in filenames:
        path = os.path.join(root, filename)
        size = os.path.getsize(path) / (1024 * 1024)
        print(f"{filename} — {size:.2f} MB")

In [ ]:
import glob
import os

RUN_DIR = "/content/drive/MyDrive/mustard_pp/runs/hubert_ctx_v2"

checkpoints = glob.glob(RUN_DIR + "/**/*", recursive=True)

for path in checkpoints:
    if os.path.isfile(path):
        print(path)

## 9. Download the model for your app
The model is also saved permanently in Drive at `RUN_DIR`. Put this zip's contents in your app and use `SarcasmAudioDetector("<folder>")`.

In [ ]:
for name, obj in list(globals().items()):
    try:
        if callable(getattr(obj, "state_dict", None)):
            print(name, type(obj))
    except Exception:
        pass


In [ ]:
for name, obj in list(globals().items()):
    if any(word in name.lower() for word in ["model", "hubert", "audio", "net"]):
        print(name, type(obj))

MODEL_NAME <class 'str'>
AUDIO_DIR <class 'str'>
SarcasmAudioDetector <class 'type'>
Audio <class 'type'>
audio <class 'numpy.ndarray'>
audio_path <class 'str'>


In [ ]:
import os

RUN_DIR = "/content/drive/MyDrive/mustard_pp/runs/hubert_ctx_v2"

for root, dirs, files in os.walk(RUN_DIR):
    for file in files:
        path = os.path.join(root, file)
        size = os.path.getsize(path) / (1024 * 1024)
        print(f"{path} — {size:.2f} MB")

In [ ]:
import os

RUN_DIR = "/content/drive/MyDrive/mustard_pp/runs/hubert_ctx_v2"

print("Folder exists:", os.path.exists(RUN_DIR))
print("Files:", os.listdir(RUN_DIR) if os.path.exists(RUN_DIR) else "Folder not found")

Folder exists: True
Files: []


In [ ]:
import os

DRIVE_DIR = "/content/drive/MyDrive/mustard_pp"

for root, dirs, files in os.walk(DRIVE_DIR):
    for file in files:
        if file.endswith((".pt", ".pth", ".ckpt", ".safetensors")):
            path = os.path.join(root, file)
            size = os.path.getsize(path) / (1024 * 1024)
            print(f"{path} — {size:.2f} MB")

/content/drive/MyDrive/mustard_pp/runs/model.pt — 362.70 MB


In [ ]:
import torch

CHECKPOINT_PATH = "/content/drive/MyDrive/mustard_pp/runs/model.pt"

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("Checkpoint keys:", checkpoint.keys())
else:
    print("Loaded object:", type(checkpoint))

Checkpoint type: <class 'collections.OrderedDict'>
Checkpoint keys: odict_keys(['layer_weights', 'backbone.masked_spec_embed', 'backbone.feature_extractor.conv_layers.0.conv.weight', 'backbone.feature_extractor.conv_layers.0.layer_norm.weight', 'backbone.feature_extractor.conv_layers.0.layer_norm.bias', 'backbone.feature_extractor.conv_layers.1.conv.weight', 'backbone.feature_extractor.conv_layers.2.conv.weight', 'backbone.feature_extractor.conv_layers.3.conv.weight', 'backbone.feature_extractor.conv_layers.4.conv.weight', 'backbone.feature_extractor.conv_layers.5.conv.weight', 'backbone.feature_extractor.conv_layers.6.conv.weight', 'backbone.feature_projection.layer_norm.weight', 'backbone.feature_projection.layer_norm.bias', 'backbone.feature_projection.projection.weight', 'backbone.feature_projection.projection.bias', 'backbone.encoder.pos_conv_embed.conv.bias', 'backbone.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'backbone.encoder.pos_conv_embed.conv.parametriz

In [ ]:
import os
import shutil

SOURCE = "/content/drive/MyDrive/mustard_pp/runs/model.pt"

SAVE_DIR = "/content/drive/MyDrive/mustard_pp/saved_models"
os.makedirs(SAVE_DIR, exist_ok=True)

DESTINATION = os.path.join(SAVE_DIR, "hubert_sarcasm_model.pt")

shutil.copy2(SOURCE, DESTINATION)

print("Model saved successfully!")
print("Location:", DESTINATION)
print("Size:", round(os.path.getsize(DESTINATION) / (1024 * 1024), 2), "MB")

Model saved successfully!
Location: /content/drive/MyDrive/mustard_pp/saved_models/hubert_sarcasm_model.pt
Size: 362.7 MB


In [ ]:
import os

ROOT = "/content/drive/MyDrive/mustard_pp"

required_files = {
    "sarcasm_audio_model.py",
    "sarcasm_audio_config.json",
    "preprocessor_config.json",
    "config.json",
    "model.pt"
}

for root, dirs, files in os.walk(ROOT):
    found = required_files.intersection(files)
    if found:
        print("\nFolder:", root)
        for file in sorted(found):
            print("  ", file)


Folder: /content/drive/MyDrive/mustard_pp/runs
   config.json
   model.pt
   preprocessor_config.json
   sarcasm_audio_config.json


In [ ]:
import os

ROOT = "/content/drive/MyDrive/mustard_pp"

for root, dirs, files in os.walk(ROOT):
    if "sarcasm_audio_model.py" in files:
        print("Found:", os.path.join(root, "sarcasm_audio_model.py"))

In [ ]:
import os

ROOT = "/content/drive/MyDrive"

for root, dirs, files in os.walk(ROOT):
    if "sarcasm_audio_model.py" in files:
        print("Found:", os.path.join(root, "sarcasm_audio_model.py"))

In [ ]:

%%writefile /content/drive/MyDrive/mustard_pp/saved_models/sarcasm_audio_model.py

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel

class SarcasmAudioModel(nn.Module):
    def __init__(self, backbone=None, config=None, num_labels=2, dropout=0.2,
                 use_context=False):
        super().__init__()

        if config is not None:
            config.layerdrop = 0.0
            self.backbone = AutoModel.from_config(config)
        else:
            self.backbone = AutoModel.from_pretrained(backbone, layerdrop=0.0)

        hidden = self.backbone.config.hidden_size
        n_layers = self.backbone.config.num_hidden_layers + 1
        self.use_context = use_context
        self.layer_weights = nn.Parameter(torch.zeros(n_layers))
        self.attn = nn.Sequential(
            nn.Linear(hidden, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        in_dim = hidden * 3 if use_context else hidden
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_labels)
        )

    def freeze_feature_encoder(self):
        fe = getattr(self.backbone, "feature_extractor", None)
        if fe is not None:
            for p in fe.parameters():
                p.requires_grad = False

    def encode(self, input_values, input_lengths, attention_mask=None):
        out = self.backbone(
            input_values,
            attention_mask=attention_mask,
            output_hidden_states=True
        )

        hs = torch.stack(out.hidden_states, dim=0)
        w = F.softmax(self.layer_weights, dim=0).view(-1, 1, 1, 1)
        x = (w.to(hs.dtype) * hs).sum(0)

        T = x.size(1)
        frames = self.backbone._get_feat_extract_output_lengths(input_lengths)
        frames = frames.clamp(min=1, max=T)

        mask = torch.arange(T, device=x.device)[None, :] < frames[:, None]
        scores = self.attn(x).squeeze(-1).float().masked_fill(~mask, -1e4)
        alpha = scores.softmax(-1).unsqueeze(-1).to(x.dtype)

        return (alpha * x).sum(1)

    def forward(self, input_values, input_lengths, attention_mask=None,
                ctx_values=None, ctx_lengths=None, ctx_attention_mask=None,
                ctx_present=None):

        u = self.encode(input_values, input_lengths, attention_mask)

        if not self.use_context:
            return self.head(u)

        if ctx_values is None:
            c = torch.zeros_like(u)
        else:
            c = self.encode(ctx_values, ctx_lengths, ctx_attention_mask)

            if ctx_present is not None:
                c = c * ctx_present.to(c.dtype)[:, None]

        return self.head(torch.cat([u, c, u * c], dim=-1))


Overwriting /content/drive/MyDrive/mustard_pp/saved_models/sarcasm_audio_model.py


In [ ]:
import os

SAVE_DIR = "/content/drive/MyDrive/mustard_pp/saved_models"

for file in os.listdir(SAVE_DIR):
    print(file)

hubert_sarcasm_model.pt
sarcasm_audio_model.py


In [ ]:
import os

MODEL_FILE = "/content/drive/MyDrive/mustard_pp/saved_models/sarcasm_audio_model.py"

with open(MODEL_FILE, "a") as f:
    f.write("""

import subprocess
import numpy as np

SAMPLE_RATE = 16000

def load_audio(path, sr=SAMPLE_RATE):
    try:
        raw = subprocess.run(
            ["ffmpeg", "-nostdin", "-v", "error", "-i", str(path),
             "-vn", "-f", "f32le", "-ac", "1", "-ar", str(sr), "-"],
            capture_output=True, check=True
        ).stdout
        wav = np.frombuffer(raw, dtype=np.float32).copy()
    except (FileNotFoundError, subprocess.CalledProcessError):
        import librosa
        wav, _ = librosa.load(str(path), sr=sr, mono=True)
        wav = wav.astype(np.float32)

    if wav.size == 0:
        raise ValueError(f"No audio decoded from {path}")

    return wav
""")

print("SAMPLE_RATE and load_audio saved successfully!")

SAMPLE_RATE and load_audio saved successfully!


In [ ]:
%%writefile /content/drive/MyDrive/mustard_pp/saved_models/sarcasm_audio_detector.py

import json
from pathlib import Path

import numpy as np
import torch
from transformers import AutoConfig, AutoFeatureExtractor

from sarcasm_audio_model import SAMPLE_RATE, SarcasmAudioModel, load_audio


def _to_wav(audio, sr):
    if isinstance(audio, (str, Path)):
        return load_audio(audio)

    wav = np.asarray(audio, dtype=np.float32)

    if wav.ndim > 1:
        wav = wav.mean(axis=-1 if wav.shape[-1] <= 8 else 0)

    if sr and sr != SAMPLE_RATE:
        import librosa
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SAMPLE_RATE)

    return wav.astype(np.float32)


class SarcasmAudioDetector:
    def __init__(self, model_dir, device=None):
        model_dir = Path(model_dir)

        with open(model_dir / "sarcasm_audio_config.json") as f:
            self.cfg = json.load(f)

        self.device = torch.device(
            device or ("cuda" if torch.cuda.is_available() else "cpu")
        )

        self.fe = AutoFeatureExtractor.from_pretrained(model_dir)

        self.use_context = bool(self.cfg.get("use_context", False))

        self.model = SarcasmAudioModel(
            config=AutoConfig.from_pretrained(model_dir),
            dropout=self.cfg.get("dropout", 0.2),
            use_context=self.use_context
        )

        state = torch.load(
            model_dir / "model.pt",
            map_location="cpu",
            weights_only=True
        )

        self.model.load_state_dict(state)
        self.model.to(self.device).eval()

        self.window = int(self.cfg.get("max_seconds", 10) * SAMPLE_RATE)
        self.ctx_keep = int(self.cfg.get("context_seconds", 8) * SAMPLE_RATE)
        self.threshold = float(self.cfg.get("threshold", 0.5))
        self._mask = True

    def _enc(self, arrs):
        e = self.fe(
            arrs,
            sampling_rate=SAMPLE_RATE,
            padding=True,
            return_tensors="pt",
            return_attention_mask=self._mask
        )

        am = e.get("attention_mask")

        return (
            e["input_values"].to(self.device),
            torch.tensor([len(a) for a in arrs], device=self.device),
            am.to(self.device) if am is not None else None
        )

    @torch.no_grad()
    def predict(self, audio, sr=None, context=None, context_sr=None):
        wav = _to_wav(audio, sr)

        hop = self.window // 2

        chunks = (
            [wav[i:i + self.window]
             for i in range(0, max(len(wav) - hop, 1), hop)]
            if len(wav) > self.window else [wav]
        )

        u = self._enc(chunks)
        used_ctx = False

        if self.use_context and context is not None:
            ctx = _to_wav(context, context_sr)[-self.ctx_keep:]
            c = self._enc([ctx] * len(chunks))
            present = torch.ones(len(chunks), device=self.device)
            logits = self.model(*u, *c, present)
            used_ctx = True
        else:
            logits = self.model(*u)

        p = float(logits.softmax(-1)[:, 1].mean())

        return {
            "label": "sarcastic" if p >= self.threshold else "not_sarcastic",
            "is_sarcastic": p >= self.threshold,
            "sarcasm_probability": round(p, 4),
            "used_context": used_ctx
        }

Writing /content/drive/MyDrive/mustard_pp/saved_models/sarcasm_audio_detector.py


In [ ]:
import os
import shutil

source = "/content/drive/MyDrive/mustard_pp/runs"
destination = "/content/drive/MyDrive/mustard_pp/saved_models"

files = [
    "model.pt",
    "config.json",
    "preprocessor_config.json",
    "sarcasm_audio_config.json"
]

for file in files:
    shutil.copy2(
        os.path.join(source, file),
        os.path.join(destination, file)
    )

print("Files copied successfully!")
print(os.listdir(destination))

Files copied successfully!
['hubert_sarcasm_model.pt', 'sarcasm_audio_model.py', 'sarcasm_audio_detector.py', 'model.pt', 'config.json', 'preprocessor_config.json', 'sarcasm_audio_config.json']


In [ ]:
import sys

MODEL_DIR = "/content/drive/MyDrive/mustard_pp/saved_models"

sys.path.insert(0, MODEL_DIR)

from sarcasm_audio_detector import SarcasmAudioDetector

detector = SarcasmAudioDetector(MODEL_DIR)

print("Model loaded successfully!")
print("Device:", detector.device)

Model loaded successfully!
Device: cuda


In [ ]:
import os
import zipfile

MODEL_DIR = "/content/drive/MyDrive/mustard_pp/saved_models"
ZIP_PATH = "/content/drive/MyDrive/mustard_pp/hubert_sarcasm_package.zip"

required_files = [
    "model.pt",
    "config.json",
    "preprocessor_config.json",
    "sarcasm_audio_config.json",
    "sarcasm_audio_model.py",
    "sarcasm_audio_detector.py"
]

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file in required_files:
        file_path = os.path.join(MODEL_DIR, file)
        if not os.path.exists(file_path):
            raise FileNotFoundError(file_path)
        zipf.write(file_path, arcname=file)

print("ZIP package created successfully!")
print("Location:", ZIP_PATH)
print("Size:", round(os.path.getsize(ZIP_PATH) / (1024**3), 2), "GB")

ZIP package created successfully!
Location: /content/drive/MyDrive/mustard_pp/hubert_sarcasm_package.zip
Size: 0.32 GB
